# Regression 2: Autograd og smarte optimizers

I regression 1 regnede I selv gradienten ud i hånden og tog ét lille skridt ned ad bakken ad gangen. Her bygger vi videre på to måder:

- **Autograd:** i stedet for at udlede gradienten selv, lader vi PyTorch finde den med `.backward()`. Så virker gradient descent på et hvilket som helst loss — også dem, I ikke kan differentiere i hånden.
- **Smarte optimizers:** når der er mange punkter (eller mange skridt), er der bedre måder at bevæge sig på end almindelig gradient descent. I bygger selv **minibatch/SGD**, **Momentum**, **RMSprop** og til sidst **ADAM** — og dyster med dem på fire landskaber.

Til allersidst får I en kort forsmag på **neurale netværk**, som trænes med præcis de samme værktøjer.


# 1: Gradienten med autograd

Vi starter med det samme datasæt som i regression 1 — bare med 100 punkter i stedet for 4. Loss er den samme (MSE for linjen $y = a\cdot x + b$), men gradienten regner vi ikke længere i hånden.

I stedet gør vi $a$ og $b$ til **tensorer**, som I kender fra intro til programmering, og beder PyTorch om gradienten med `.backward()`:


In [ ]:
# Henter hjælpefunktioner + testfil fra GitHub (Plan B: upload dem manuelt via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/02-Regression/regression2/regression2_test.py

import numpy as np
import helpers as sesy_viz          # sesy_vizualisation er foldet ind i helpers.py
import helpers as tk                # regression2_helper (LINJE_PUNKTER, LANDSKABER) er også i helpers.py

In [ ]:
import torch

punkter = tk.LINJE_PUNKTER   # ~100 punkter, normalfordelt omkring en linje

# MSE for linjen y = a*x + b (samme loss som i regression 1)
def loss(a, b, punkter):
    n = len(punkter)
    total = 0
    for x, y in punkter:
        total += (a*x + b - y)**2
    return total / n

# gradienten af loss - nu med autograd i stedet for i hånden:
# gør a og b til tensorer, kald loss(a, b).backward(), og aflæs .grad
def gradient(a, b, punkter):
    a = torch.tensor(float(a), dtype=torch.float64, requires_grad=True)
    b = torch.tensor(float(b), dtype=torch.float64, requires_grad=True)
    loss(a, b, punkter).backward()
    return a.grad.item(), b.grad.item()

# ét gradient descent-skridt (som i regression 1)
def skridt(a, b, punkter, lr):
    da, db = gradient(a, b, punkter)
    return a - lr*da, b - lr*db

Her er en illustration af losslandskabet og af, hvordan gradient descent træner med alle 100 punkter:


In [ ]:

a, b = 0.0, 0.0
lr = 0.02

sti = [(a, b)]
paneler = []
for _ in range(50):
    a, b = sti[-1]
    grad_a, grad_b = gradient(a, b, punkter)
    paneler.append((
        sesy_viz.modelfit(a, b, punkter, x_range=(0,10), y_range=(-2,14)),
        sesy_viz.loss_kontur(lambda a, b: loss(a, b, punkter), a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(sti), gradient=(grad_a, grad_b), colorbar=False),
        sesy_viz.loss_over_tid([loss(a, b, punkter) for a, b in sti]),
    ))
    sti.append(skridt(a, b, punkter, lr))

print("Beregning færdig, samler til animation:")
sesy_viz.animate(paneler, max_frames=25)

# 2: Minibatch og stokastisk gradient descent

Hvorfor regne gradienten ud fra ALLE punkter for hvert skridt, hvis vi kan nøjes med et tilfældigt udpluk?

Med 100 punkter (og for rigtige datasæt ofte millioner) er det dyrt at bruge alle punkter hvert skridt. Løsningen er at trække et tilfældigt udpluk — en **batch** — og regne gradienten ud fra den. Den gradient peger næsten samme vej som gradienten fra alle punkter, men er meget billigere. Det kaldes **minibatch** eller **stokastisk gradient descent (SGD)**.

### Opgaver

##### Opgave 1.1

Skriv `opg1_1(punkter, k)`, der trækker `k` tilfældige punkter UDEN tilbagelægning fra `punkter`.

##### Opgave 1.2

Skriv træningsløkken `opg1_2`: træk en ny batch med `opg1_1`, og tag ét `skridt` (fra afsnit 1) — `n` gange i træk.


In [ ]:
from regression2_test import *
import random

# træk k tilfældige punkter, UDEN tilbagelægning, fra en liste af punkter
def opg1_1(punkter, k):
    return random.sample(punkter, k)
testopg1_1(opg1_1)

# træningsloop: træk en ny batch (opg1_1) og tag ét skridt (skridt, fra recap), n gange i træk
def opg1_2(a, b, punkter, batch_størrelse, lr, n):
    for _ in range(n):
        batch = opg1_1(punkter, batch_størrelse)
        a, b = skridt(a, b, batch, lr)
    return a, b
testopg1_2(opg1_2)

Illustrationen viser, hvordan loss-konturen ændrer sig med forskellige batch-størrelser, og hvordan SGD-stien ser ud:


In [ ]:
batch_størrelse = 5   # prøv fx 1 (meget jitter) eller 50 (næsten roligt) og kør cellen igen

full_loss_fn = lambda a, b: loss(a, b, punkter)   # landskabet regnet ud fra ALLE punkter — stabilt facit-landskab

a, b = 0.0, 0.0
lr = 0.02

sti = [(a, b)]
paneler = []
for _ in range(100):
    a, b = sti[-1]
    batch = opg1_1(punkter, batch_størrelse)             # jeres egen funktion
    grad_a, grad_b = gradient(a, b, batch)                # kun til at tegne pilen — selve skridtet tages nedenfor
    batch_loss_fn = lambda a, b, batch=batch: loss(a, b, batch)   # landskabet regnet KUN ud fra denne batch
    paneler.append((
        sesy_viz.modelfit(a, b, punkter, x_range=(0,10), y_range=(-2,14)),
        sesy_viz.loss_kontur(full_loss_fn, a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(sti), gradient=(grad_a, grad_b), colorbar=False, titel='loss(alle punkter)'),
        sesy_viz.modelfit(a, b, batch, x_range=(0,10), y_range=(-2,14)),
        sesy_viz.loss_kontur(batch_loss_fn, a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(sti), gradient=(grad_a, grad_b), colorbar=False, titel='loss(kun denne batch)'),
    ))
    sti.append(skridt(a, b, batch, lr))   # skridt fra recap, PÅ SAMME batch

print("Beregning færdig, samler til animation:")
sesy_viz.animate(paneler, max_frames=50)

##### Opgave 1.3

Prøv `batch_størrelse` = 1, 5, 20, 50, 100 i cellen ovenfor, og kør den igen. Beskriv med egne ord, hvad der ændrer sig:


In [ ]:
# beskriv med jeres egne ord hvad der sker ved forskellige batch-størrelser
opg1_3 = """
...
"""
testopg1_3(opg1_3)

# 3: Smarte optimizers — en lille konkurrence

Kan vi komme hurtigere og mere direkte til minimum, uden at bruge flere punkter pr. skridt? Det er dét, metoder som **ADAM** — en af de mest brugte optimizers overhovedet — forsøger.

I stedet for at gennemgå ADAM slavisk, gør vi det til en lille konkurrence. I skriver en funktion `metode(loss, start, ...)`, der prøver at finde punktet med lavest loss.

**Reglerne:**
- I får kun `loss(a, b)` — ikke gradienten. Den finder I selv med `.backward()` (se `gradienten` nedenfor).
- I må kalde `loss` så mange gange, I vil, men vi stopper jer efter **100** kald tilsammen, så metoderne kan sammenlignes retfærdigt.
- I må gerne stoppe og returnere før budgettet er brugt, hvis I er tætte nok på.
- Bruger I alle 100 kald uden selv at returnere, bruges automatisk det sidste punkt, I nåede at undersøge (I får en advarsel).
- Beder I om et punkt meget langt fra landskabet (over 1000 væk), stopper det på samme måde — det er nok en fejl (fx et alt for stort skridt).

Sammenligningen køres på **4 landskaber**: `linjefitting` (som I kender) og 3 "underlige" landskaber, I ikke kender formlen for — og fra 4 forskellige startpunkter, valgt så metoderne opfører sig så forskelligt som muligt.

I bygger `gd_metode`, og siden `momentum_metode`, `rmsprop_metode` og `adam_metode` — hver i to lag: selve opdateringen (`..._skridt`, testet for sig) og løkken udenom (`..._metode`, testet som helhed). Værktøjer fra `tk`:
- `test_..._skridt(...)` og `test_..._metode(...)` tjekker, om jeres formel er rigtig.
- `tk.evaluer_gd_metode([("gd", gd_metode), ...])` viser metoderne side om side: et søjlediagram pr. landskab og hver metodes sti fra de 4 startpunkter. Skriv hele listen igen, hver gang I vil sammenligne.


In [ ]:
# I konkurrencen får I KUN loss(a, b) - ikke gradienten. Men den finder I selv med autograd:
# gør a og b til tensorer, kald loss(a, b).backward(), og aflæs .grad. Samme .backward()-trick
# som i afsnit 1, nu for et hvilket som helst loss.
def gradienten(loss, a, b):
    a = torch.tensor(float(a), dtype=torch.float64, requires_grad=True)
    b = torch.tensor(float(b), dtype=torch.float64, requires_grad=True)
    loss(a, b).backward()
    return a.grad.item(), b.grad.item()

## Almindelig gradient descent

Den simpleste metode: find gradienten, og tag ét skridt imod den — igen og igen.


In [ ]:
from regression2_test import *

# ét gradient descent-skridt: find gradienten af loss med .backward(), og træk lr*gradienten fra
def gd_skridt(a, b, loss, lr):
    da, db = gradienten(loss, a, b)
    return a - lr*da, b - lr*db
test_gd_skridt(gd_skridt)

# løkken udenom: kald gd_skridt n gange i træk
def gd_metode(loss, start, lr=0.01, n=100):
    a, b = start
    for _ in range(n):
        a, b = gd_skridt(a, b, loss, lr)
    return a, b
test_gd_metode(gd_metode)

tk.evaluer_gd_metode([("gd", gd_metode)])

**Frit spil:** bemærk især hvor galt det går på `rosenbrock` — prøv at forbedre den, som
`egen_gd_metode_v2` (andet `lr`, eller en helt anden idé). Ingen facit at teste imod her —
tilføj jeres variant til listen og sammenlign direkte med `tk.evaluer_gd_metode(...)`:


In [ ]:
def egen_gd_metode_v1(loss, start, lr=0.02, n=100):
    a, b = start
    da, db = gradienten(loss, a, b)
    for _ in range(n):
        a, b = a - lr * da, b - lr * db
    return a, b

tk.evaluer_gd_metode([
    ("gd", gd_metode),
    ("egen v1", egen_gd_metode_v1),
])

## Momentum

Momentum er IKKE en fysisk "bold der ruller ned ad bakken" — det er en **vægtet, aftagende
gennemsnit af alle de gradienter man har set indtil videre**:

$$v_t = (\beta) \cdot v_{t-1} + (1-\beta)\cdot \nabla L(a_t, b_t), \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot v_t$$

$\beta$ (typisk omkring 0.9) styrer hvor meget vægt de ældre gradienter stadig har — en
gradient fra $k$ skridt tilbage tæller med vægten $(1-\beta)\beta^k$, altså mindre og mindre
jo længere tilbage den er. Bygges i to lag, ligesom `gd_metode`: `momentum_v_opdater` (selve
$v$-formlen) og `momentum_skridt` (ét skridt, som bruger $v$ i stedet for selve gradienten).

In [ ]:
from regression2_test import *

# opdater det glidende gennemsnit v, med den nye gradient vægtet ind
def momentum_v_opdater(va, vb, da, db, beta):
    return beta*va + (1-beta)*da, beta*vb + (1-beta)*db
test_momentum_v_opdater(momentum_v_opdater)

# ét momentum-skridt: find gradienten, opdater v, brug SÅ v (ikke selve gradienten) til skridtet
def momentum_skridt(a, b, va, vb, loss, lr, beta):
    da, db = gradienten(loss, a, b)
    va, vb = momentum_v_opdater(va, vb, da, db, beta)
    return a - lr*va, b - lr*vb, va, vb
test_momentum_skridt(momentum_skridt)

# løkken udenom: kald momentum_skridt n gange i træk, og hold styr på v undervejs
def momentum_metode(loss, start, lr=0.02, beta=0.9, n=99):
    a, b = start
    va, vb = 0.0, 0.0
    for _ in range(n):
        a, b, va, vb = momentum_skridt(a, b, va, vb, loss, lr, beta)
    return a, b
test_momentum_metode(momentum_metode)

Se momentum illustreret med jeres eget `momentum_skridt` kaldt for hvert billede: 
* venstre panel er gradienten lige nu (+ alle tidligere målinger, gråt), 
* midten samler dem til $v$ (ældre vejer mindre),
* højre er det skridt der rent faktisk tages:

In [ ]:
landskab = [l for l in tk.LANDSKABER if l["navn"] == "plateau (1D)"][0]
lr, beta = 0.1, 0.9
vis_skala = 1.0   # ren visningsskala for pilene — juster hvis de er for små/store til at se

a, b = 1, 0.6
va, vb = 0.0, 0.0
referencer = []   # hver tidligere gradients bidrag (1-beta)*grad, DA den var ny
målinger = []     # (a, b, ref_a, ref_b) — hvor + hvad, til panel 1's grå pile
sti = [(a, b)]

frames = []
for _ in range(100):
    da, db = gradienten(landskab["loss"], a, b)   # kun til visning i panel 1 (selve skridtet tages nedenfor)
    reference = (vis_skala * (1 - beta) * da, vis_skala * (1 - beta) * db)
    referencer.append(reference)
    målinger.append((a, b, *reference))

    # ét par pr. tidligere gradient: (reference = værdien DA den kom) og (nu = dens aktuelle,
    # vægtede bidrag til v_t — reference skrumpet med beta**alder)
    n = len(referencer)
    par = [(ref, (ref[0] * beta**alder, ref[1] * beta**alder))
           for alder, ref in zip(range(n - 1, -1, -1), referencer)]

    ny_a, ny_b, va, vb = momentum_skridt(a, b, va, vb, landskab["loss"], lr, beta)   # jeres egen funktion

    fælles = dict(path=list(sti), colorbar=False)
    frames.append((
        sesy_viz.loss_kontur(landskab["loss"], landskab["a_range"], landskab["b_range"], gradient=(vis_skala * da, vis_skala * db),
                              ekstra_pile=[(ma, mb, mda, mdb, "gray", 0.5) for ma, mb, mda, mdb in målinger],
                              titel="gradient nu (+ tidligere målinger)", **fælles),
        sesy_viz.vektor_vifte(par, sum_vektor=(vis_skala * va, vis_skala * vb), titel="tidligere gradienter: original vs. vægtet"),
        sesy_viz.loss_kontur(landskab["loss"], landskab["a_range"], landskab["b_range"], gradient=(vis_skala * va, vis_skala * vb),
                              gradient_farve="orange", titel="momentum-skridt", **fælles),
    ))
    a, b = ny_a, ny_b
    sti.append((a, b))

sesy_viz.animate(frames, max_frames=50)

**Frit spil:** prøv jeres egen variant af momentum — fx et andet `beta`, eller en helt anden
idé for hvordan I bruger de tidligere gradienter. Tilføj den til listen og sammenlign:

In [ ]:
def egen_gd_metode_v2(loss, start, lr=0.02, n=100):
    a, b = start
    da, db = gradienten(loss, a, b)
    for _ in range(n):
        a, b = a - lr * da, b - lr * db
    return a, b

tk.evaluer_gd_metode([
    ("gd", gd_metode),
    ("momentum", momentum_metode),
    ("egen v2", egen_gd_metode_v2),
])

## RMSprop

RMSprops idé: skalér hver akse med sin egen (nyligt sete) gradient-størrelse, så gradienten i den skalerede akse ligger omkring $\pm 1$ .
ie vi optimere ikke bare en værdi ad gangen, som nogen gange sker med den normale gradient.
dermed tages der lige store skridt i alle retninger, uanset om den oprindelige akse var stejl eller flad:

$$s_t = \beta\cdot s_{t-1}+(1-\beta)\cdot \nabla L(a_t,b_t)^2, \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot\frac{\nabla L(a_t,b_t)}{\sqrt{s_t}+\varepsilon}$$

($s$ regnes ELEMENTVIS pr. parameter — $a$ og $b$ får hver deres egen skalering. $\varepsilon$
er blot et lille tal, der forhindrer division med 0.) Samme to-lags opskrift som momentum:
`rmsprop_s_opdater` (selve $s$-formlen) og `rmsprop_skridt` (ét skridt).

In [ ]:
from regression2_test import *

# opdater det glidende gennemsnit s af GRADIENTEN I ANDEN
def rmsprop_s_opdater(sa, sb, da, db, beta):
    return beta*sa + (1-beta)*da**2, beta*sb + (1-beta)*db**2
test_rmsprop_s_opdater(rmsprop_s_opdater)

# ét rmsprop-skridt: find gradienten, opdater s, og skalér gradienten med sqrt(s) før skridtet
def rmsprop_skridt(a, b, sa, sb, loss, lr, beta, eps):
    da, db = gradienten(loss, a, b)
    sa, sb = rmsprop_s_opdater(sa, sb, da, db, beta)
    return a - lr*da/(sa**0.5+eps), b - lr*db/(sb**0.5+eps), sa, sb
test_rmsprop_skridt(rmsprop_skridt)

# løkken udenom: kald rmsprop_skridt n gange i træk, og hold styr på s undervejs
def rmsprop_metode(loss, start, lr=0.02, beta=0.5, eps=1e-8, n=99):
    a, b = start
    sa, sb = 0.0, 0.0
    for _ in range(n):
        a, b, sa, sb = rmsprop_skridt(a, b, sa, sb, loss, lr, beta, eps)
    return a, b
test_rmsprop_metode(rmsprop_metode)

se RMSprop animeret med jeres egen `rmsprop_skridt` kaldt for hvert billede: venstre panel er den rå gradient, midten er det SAMME rum omskaleret med $\sqrt{s_t}$ (en aflang dal bliver rund), højre er skridtet der rent faktisk tages:

In [ ]:
landskab = [l for l in tk.LANDSKABER if l["navn"] == "linjefitting"][0]
lr, beta, eps = 0.02, 0.1, 1e-8   # I har selv valgt disse (ikke rmsprop_metode's defaults)
vis_skala = 0.01   # visningsskala for den RÅ gradient-pil (den rmsprop-skalerede pil er allerede ca. i størrelsesorden 1)
farve_range = sesy_viz.loss_range(landskab["loss"], landskab["a_range"], landskab["b_range"])   # samme farveskala i alle paneler/billeder
a, b = 0, 0
sa, sb = 0.0, 0.0
sti = [(a, b)]

frames = []
for _ in range(100):
    da, db = gradienten(landskab["loss"], a, b)   # kun til visning i panel 1
    ny_a, ny_b, sa, sb = rmsprop_skridt(a, b, sa, sb, landskab["loss"], lr, beta, eps)   # jeres egen funktion
    skala_a, skala_b = sa**0.5 + eps, sb**0.5 + eps
    vis_a, vis_b = da / skala_a, db / skala_b   # den skalerede retning — det panel 2/3 viser

    # panel 2's a-akse beholder landskabets EGEN, faste range hele vejen igennem (aldrig
    # zoomet ind/ud pr. billede) — så I kan se rummet selv strække/klemme sig (aksetallene
    # ændrer sig med skala_a: store tal tidligt, tæt på 1 midtvejs, små sent). b-aksens
    # BREDDE sættes til at matche a-aksens (samme antal enheder i det omskalerede rum — et
    # ægte KVADRAT, ikke et skævt rektangel), men centreret om nuværende b (ikke om 0), så
    # det er dér kurven rent faktisk bøjer der vises.
    bredde = (landskab["a_range"][1] - landskab["a_range"][0]) * skala_a
    lokal_b_range = (b - bredde / (2 * skala_b), b + bredde / (2 * skala_b))

    fælles = dict(path=list(sti), colorbar=False, farve_range=farve_range)
    frames.append((
        sesy_viz.loss_kontur(landskab["loss"], landskab["a_range"], landskab["b_range"], gradient=(vis_skala * da, vis_skala * db), titel="gradient nu", **fælles),
        sesy_viz.loss_kontur(landskab["loss"], landskab["a_range"], lokal_b_range, skala=(skala_a, skala_b),
                              gradient=(vis_skala * da, vis_skala * db),
                              ekstra_pile=[(a, b, vis_a, vis_b, "orange", 1.0)],
                              titel="skaleret rum", **fælles),
        sesy_viz.loss_kontur(landskab["loss"], landskab["a_range"], landskab["b_range"], gradient=(vis_a, vis_b),
                              gradient_farve="orange", titel="rmsprop-skridt", **fælles),
    ))
    a, b = ny_a, ny_b
    sti.append((a, b))

sesy_viz.animate(frames, max_frames=50)

**Frit spil:** prøv jeres egen variant af RMSprop — fx et andet `beta`/`lr`, eller kombinér
med en idé fra momentum. Tilføj den til listen og sammenlign:

In [ ]:
def egen_gd_metode_v3(loss, start, lr=0.02, n=100):
    a, b = start
    da, db = gradienten(loss, a, b)
    for _ in range(n):
        a, b = a - lr * da, b - lr * db
    return a, b

tk.evaluer_gd_metode([
    ("gd", gd_metode),
    ("momentum", momentum_metode),
    ("rmsprop", rmsprop_metode),
    ("egen v3", egen_gd_metode_v3),
])

## ADAM

Adam kombinerer de to metoder ovenfor: $v_t$ fra momentum og $s_t$ fra RMSprop.
I kan genbruger jeres `momentum_v_opdater` og `rmsprop_s_opdater` fra før.
plus en ny **bias-korrektion** af begge, vigtig i de første skridt, hvor $v_0=s_0=0$ ellers trækker
gennemsnittet kunstigt mod nul:

$$v_t = \beta_1 v_{t-1} + (1-\beta_1)\nabla L(a_t,b_t), \qquad s_t = \beta_2 s_{t-1} + (1-\beta_2)\nabla L(a_t,b_t)^2$$

$$\hat v_t = \frac{v_t}{1-\beta_1^t}, \qquad \hat s_t = \frac{s_t}{1-\beta_2^t}, \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot\frac{\hat v_t}{\sqrt{\hat s_t}+\varepsilon}$$

In [ ]:
from regression2_test import *

# ét adam-skridt: samme v/s-opdatering som momentum/rmsprop (genbrugt!), plus bias-korrektion af begge
def adam_skridt(a, b, va, vb, sa, sb, t, loss, lr, beta1, beta2, eps):
    da, db = gradienten(loss, a, b)
    va, vb = momentum_v_opdater(va, vb, da, db, beta1)
    sa, sb = rmsprop_s_opdater(sa, sb, da, db, beta2)
    vha, vhb = va / (1 - beta1**t), vb / (1 - beta1**t)
    sha, shb = sa / (1 - beta2**t), sb / (1 - beta2**t)
    return a - lr*vha/(sha**0.5+eps), b - lr*vhb/(shb**0.5+eps), va, vb, sa, sb
test_adam_skridt(adam_skridt)

# løkken udenom: kald adam_skridt n gange i træk - t starter ved 1 (bruges i bias-korrektionen)
def adam_metode(loss, start, lr=0.2, beta1=0.9, beta2=0.99, eps=1e-8, n=99):
    a, b = start
    va, vb = 0.0, 0.0
    sa, sb = 0.0, 0.0
    for t in range(1, n + 1):
        a, b, va, vb, sa, sb = adam_skridt(a, b, va, vb, sa, sb, t, loss, lr, beta1, beta2, eps)
    return a, b
test_adam_metode(adam_metode)

# alt I har bygget i denne blok, samlet i ét sidste overblik:
tk.evaluer_gd_metode([
    ("gd", gd_metode),
    ("momentum", momentum_metode),
    ("rmsprop", rmsprop_metode),
    ("adam", adam_metode),
])

se adam animeret, med jeres egen `adam_skridt` kaldt for hvert billede.
- panel 1 er den originale gradient
- panel 2 er $\hat v_t$ (momentum-viften, korrigeret), 
- panel 3 er samme $\hat v_t$ (orange) skaleret ned med $\sqrt{\hat s_t}$ til en ny, magenta pil - den pil er adam-skridtet i panel 4:

In [ ]:
landskab = [l for l in tk.LANDSKABER if l["navn"] == "rosenbrock"][0]
lr, beta1, beta2, eps = 0.2, 0.9, 0.99, 1e-8   # højere lr og lavere beta2 end adam_metode's
                                                # defaults — kun her, så begge korrektioner ses
                                                # tydeligt inden for 80 billeder
vis_skala = 0.2   # visningsskala for den rå gradient/v̂-pil (v̂/ŝ er allerede ca. størrelsesorden 1)
farve_range = sesy_viz.loss_range(landskab["loss"], landskab["a_range"], landskab["b_range"])
a, b = -1.0, -0.5
va, vb = 0.0, 0.0
sa, sb = 0.0, 0.0
historik = []   # (da, db, ref_a, ref_b) — reference frosset fra dengang gradienten blev målt
målinger = []   # (a, b, ref_a, ref_b), til panel 1's grå pile
sti = [(a, b)]

frames = []
for t in range(1, 101):
    da, db = gradienten(landskab["loss"], a, b)   # kun til visning i panel 1
    korr1 = 1 / (1 - beta1**t)
    reference = (korr1 * vis_skala * (1 - beta1) * da, korr1 * vis_skala * (1 - beta1) * db)
    historik.append((da, db, *reference))
    målinger.append((a, b, *reference))

    ny_a, ny_b, va, vb, sa, sb = adam_skridt(a, b, va, vb, sa, sb, t, landskab["loss"], lr, beta1, beta2, eps)   # jeres egen funktion
    vha, vhb = va / (1 - beta1**t), vb / (1 - beta1**t)
    skala_a, skala_b = (sa / (1 - beta2**t))**0.5 + eps, (sb / (1 - beta2**t))**0.5 + eps

    # samme vifte-konstruktion som momentum: reference er frosset (med SIN egen korr1 fra
    # dengang) — kun "nu"-bidraget bruger dagens korr1
    n = len(historik)
    par = [((ref_a, ref_b), (korr1 * vis_skala * (1 - beta1) * beta1**alder * hda, korr1 * vis_skala * (1 - beta1) * beta1**alder * hdb))
           for alder, (hda, hdb, ref_a, ref_b) in zip(range(n - 1, -1, -1), historik)]

    v_vis_a, v_vis_b = vis_skala * vha, vis_skala * vhb   # v̂ (orange) — samlet i panel 2, bæres videre til panel 3
    sv_vis_a, sv_vis_b = vha / skala_a, vhb / skala_b     # v̂/ŝ (magenta) — nyt i panel 3, bæres videre til panel 4

    # a-aksen beholder landskabets EGEN, faste range hele vejen (samme idé som RMSprop-
    # panelet ovenfor) — b-aksens bredde matcher a's (et ægte kvadrat), centreret om
    # nuværende b.
    bredde = (landskab["a_range"][1] - landskab["a_range"][0]) * skala_a
    lokal_b_range = (b - bredde / (2 * skala_b), b + bredde / (2 * skala_b))

    fælles = dict(path=list(sti), colorbar=False)
    frames.append((
        sesy_viz.loss_kontur(landskab["loss"], landskab["a_range"], landskab["b_range"], gradient=(vis_skala * da, vis_skala * db),
                              ekstra_pile=[(ma, mb, mda, mdb, "gray", 0.5) for ma, mb, mda, mdb in målinger],
                              titel="gradient nu (+ tidligere målinger)", farve_range=farve_range, **fælles),
        sesy_viz.vektor_vifte(par, sum_vektor=(v_vis_a, v_vis_b), titel="v (korrigeret): momentum-vifte"),
        sesy_viz.loss_kontur(landskab["loss"], landskab["a_range"], lokal_b_range, skala=(skala_a, skala_b),
                              gradient=(v_vis_a, v_vis_b), gradient_farve="orange",
                              ekstra_pile=[(a, b, sv_vis_a, sv_vis_b, "magenta", 1.0)],
                              titel="s (korrigeret): skaleret rum", farve_range=farve_range, **fælles),
        sesy_viz.loss_kontur(landskab["loss"], landskab["a_range"], landskab["b_range"],
                              gradient=(sv_vis_a, sv_vis_b), gradient_farve="magenta", titel="adam-skridt", farve_range=farve_range, **fælles),
    ))
    a, b = ny_a, ny_b
    sti.append((a, b))

sesy_viz.animate(frames, max_frames=50)

**Sidste frit spil:** byg jeres bedste mulige metode — kombinér gerne idéer fra gd/momentum/
rmsprop/adam, eller prøv noget helt nyt. Tilføj den til listen sammen med alle de andre, og se
om I kan slå dem alle:

In [ ]:
def egen_bedste_metode(loss, start, lr=0.01, n=100):
    a, b = start
    for _ in range(n):
        a, b = gd_skridt(a, b, loss, lr)
    return a, b

tk.evaluer_gd_metode([
    ("gd", gd_metode),
    ("momentum", momentum_metode),
    ("rmsprop", rmsprop_metode),
    ("adam", adam_metode),
    ("min bedste", egen_bedste_metode),
])

# 4: Neurale netværk — en forsmag

ADAM er ikke kun til to tal, $a$ og $b$. Det er præcis den optimizer, der træner rigtige **neurale netværk** med tusindvis af tal — og autograd (jeres `.backward()`) er dét, der finder alle gradienterne, uanset hvor stort netværket er.

Her er et lillebitte neuralt netværk, der lærer en kurve. Læg mærke til, at træningsløkken er den samme rytme som i konkurrencen: nulstil gradienter, `.backward()`, tag et skridt. I bygger dem for alvor i *Intro til Machine Learning* — dette er kun en forsmag, så I har set det før.


In [ ]:
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(42)

# et lille datasæt: en kurve (ikke en ret linje), så vi kan se netværket bøje sig
X = torch.linspace(-3, 3, 80).reshape(-1, 1)
y = torch.sin(X) + 0.1 * torch.randn(X.shape)

# et netværk: 1 tal ind -> skjulte neuroner -> 1 tal ud
class LilleNet(nn.Module):
    def __init__(self, skjulte=16):
        super().__init__()
        self.lag1 = nn.Linear(1, skjulte)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(skjulte, 1)

    def forward(self, x):
        x = self.lag1(x)
        x = self.aktivering(x)
        x = self.lag2(x)
        return x

model = LilleNet(skjulte=16)
tabsfunktion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

tab_historik = []
for epoke in range(800):
    optimizer.zero_grad()          # 1. nulstil gamle gradienter
    y_hat = model(X)               # 2. forward: netværkets gæt
    tab = tabsfunktion(y_hat, y)   # 3. hvor forkert er gættet?
    tab.backward()                 # 4. autograd finder ALLE gradienter
    optimizer.step()               # 5. ADAM tager ét skridt
    tab_historik.append(tab.item())

print("sluttab:", round(tab_historik[-1], 4))

Netværket fitter kurven, og loss falder undervejs:


In [ ]:
with torch.no_grad():
    y_forudsagt = model(X)

xs = X.reshape(-1).numpy()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(xs, y.reshape(-1).numpy(), s=12, label="data")
ax1.plot(xs, y_forudsagt.reshape(-1).numpy(), color="orange", linewidth=2, label="netværket")
ax1.set_title("Netvaerkets kurve"); ax1.legend()
ax2.plot(tab_historik)
ax2.set_title("Loss over tid"); ax2.set_xlabel("epoke"); ax2.set_ylabel("tab")
plt.tight_layout(); plt.show()

### Opgaver

##### Opgave 4.1

Prøv at ændre `skjulte=16` (i `LilleNet(skjulte=...)`) til fx `2` eller `64`, træn igen, og se, hvordan kurven og slut-tabet ændrer sig. Hvad sker der, når der er for få neuroner til at følge kurven?


In [ ]:
# prøv fx skjulte=2 (for få) eller skjulte=64 (mange), og sammenlign slut-tabet
model = LilleNet(skjulte=64)
tabsfunktion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

tab_historik = []
for epoke in range(800):
    optimizer.zero_grad()
    tab = tabsfunktion(model(X), y)
    tab.backward()
    optimizer.step()
    tab_historik.append(tab.item())

print("sluttab med 64 skjulte:", round(tab_historik[-1], 4))

# Godt gået!

I har nu bygget gradient descent oven på autograd, dystet med Momentum, RMSprop og ADAM på fire landskaber, og set jeres første neurale netværk trænet med præcis de samme værktøjer.

**Næste stop:** *Intro til Machine Learning*, hvor I bygger neurale netværk for alvor.
